# Chapter 2 examples: the semiring family

This notebook runs the examples of
[Chapter 2](https://thduynguyen.github.io/gtsam/chapter02). One short
variable-elimination routine is run on the track of Chapter 1 with five
different semirings, and each answers a different question.

GTSAM Copyright 2010, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/thduynguyen/gtsam/blob/feature/semiringfactor/gtsam/semiring/doc/chapter02_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    # This notebook needs only numpy, scipy and plotly, which Colab provides.
    %pip install --quiet numpy scipy plotly
except ImportError:
    pass  # Not running on Colab, do nothing

In [2]:
import numpy as np
from scipy.special import logsumexp, softmax

np.set_printoptions(precision=4, suppress=True)

## The track of Chapter 1

Three cells, two moves. A table has one axis per variable.

In [3]:
L, R = 0, 1
prior = np.array([0.5, 0.5, 0.0])  # p(s0)
policy = np.full((3, 2), 0.5)  # pi(a | s): a coin flip
dynamics = np.zeros((3, 2, 3))  # p(s' | s, a)
dynamics[0, L], dynamics[0, R] = [1, 0, 0], [0.2, 0.8, 0]
dynamics[1, L], dynamics[1, R] = [0.8, 0.2, 0], [0, 0.2, 0.8]
dynamics[2, L], dynamics[2, R] = [0, 0.8, 0.2], [0, 0, 1]
reward = np.array([[0.0, -1.0]] * 3)  # r(s, a): moving right costs 1
final = np.array([0.0, 0.0, 10.0])  # r(s2): the charger is in cell 2

# The eight terms of the two-move problem: (kind, variables, table).
terms = [
    ("probability", ("s0",), prior),
    ("probability", ("s0", "a0"), policy),
    ("reward", ("s0", "a0"), reward),
    ("probability", ("s0", "a0", "s1"), dynamics),
    ("probability", ("s1", "a1"), policy),
    ("reward", ("s1", "a1"), reward),
    ("probability", ("s1", "a1", "s2"), dynamics),
    ("reward", ("s2",), final),
]
order = ["s2", "a1", "s1", "a0", "s0"]  # backward in time

## Five semirings (Section 2)

An entry of a factor is a tuple of numbers, its *channels*. A semiring says
how to lift a probability and a reward to an entry, how to multiply two
entries, and how to add entries over the values of a variable.

In [4]:
class SumProduct:
    """One channel, a probability: ordinary elimination."""
    name = "sum-product"
    probability = staticmethod(lambda p: (p,))
    reward = staticmethod(lambda r: (np.ones_like(r),))  # rewards are ignored
    times = staticmethod(lambda a, b: (a[0] * b[0],))
    plus = staticmethod(lambda a, axis: (a[0].sum(axis),))
    read = staticmethod(lambda a: a[0])


class Expectation:
    """Probability and weighted value (p, w), w = p v: Chapter 1."""
    name = "expectation"
    probability = staticmethod(lambda p: (p, np.zeros_like(p)))
    reward = staticmethod(lambda r: (np.ones_like(r), r))
    times = staticmethod(lambda a, b: (a[0] * b[0], a[0] * b[1] + a[1] * b[0]))
    plus = staticmethod(lambda a, axis: (a[0].sum(axis), a[1].sum(axis)))
    read = staticmethod(lambda a: a[1] / a[0])


class MaxSum:
    """One channel, a value: values add, and the best outcome is kept."""
    name = "max-sum"
    # A possible outcome contributes 0, an impossible one minus infinity.
    probability = staticmethod(
        lambda p: (np.where(p > 0, 0.0, -np.inf),))
    reward = staticmethod(lambda r: (r,))
    times = staticmethod(lambda a, b: (a[0] + b[0],))
    plus = staticmethod(lambda a, axis: (a[0].max(axis),))
    read = staticmethod(lambda a: a[0])


class Tilted:
    """Probability and tilted value (p, m), m = p exp(kappa v)."""

    def __init__(self, kappa):
        self.kappa, self.name = kappa, f"tilted, kappa = {kappa:g}"

    def probability(self, p):
        return (p, p)

    def reward(self, r):
        return (np.ones_like(r), np.exp(self.kappa * r))

    def times(self, a, b):
        return (a[0] * b[0], a[1] * b[1])

    def plus(self, a, axis):
        return (a[0].sum(axis), a[1].sum(axis))

    def read(self, a):
        return np.log(a[1] / a[0]) / self.kappa

## One elimination routine for all of them (Section 1)

This is bucket elimination: for each variable, multiply the factors that
contain it, add over its values, and put the new factor back.

In [5]:
def expand(factor, variables):
    """Reshape the channels of a factor so its axes follow `variables`."""
    names, channels = factor
    index = [names.index(v) for v in variables if v in names]
    shape = [-1 if v in names else 1 for v in variables]
    return tuple(np.transpose(c, index).reshape(
        [c.shape[names.index(v)] if v in names else 1 for v in variables])
        for c in channels)


def multiply(semiring, factors):
    """The product of several factors, on the union of their variables."""
    variables = tuple(dict.fromkeys(v for names, _ in factors for v in names))
    product = expand(factors[0], variables)
    for factor in factors[1:]:
        product = semiring.times(product, expand(factor, variables))
    shape = np.broadcast_shapes(*(c.shape for c in product))
    return variables, tuple(np.broadcast_to(c, shape) for c in product)


def eliminate(semiring, terms, order):
    """Variable elimination. Returns what is left when no variable remains."""
    factors = [(names, getattr(semiring, kind)(np.asarray(table, float)))
               for kind, names, table in terms]
    for variable in order:
        bucket = [f for f in factors if variable in f[0]]
        factors = [f for f in factors if variable not in f[0]]
        names, product = multiply(semiring, bucket)
        axis = names.index(variable)
        separator = tuple(v for v in names if v != variable)
        factors.append((separator, semiring.plus(product, axis)))
    return multiply(semiring, factors)[1]


for semiring in [SumProduct(), Expectation(), MaxSum()]:
    result = eliminate(semiring, terms, order)
    print(f"{semiring.name:12s} {float(semiring.read(result)):8.4f}")

sum-product    1.0000
expectation    1.4000
max-sum        9.0000


Sum-product gives 1: the probabilities of all trajectories sum to one.
The expectation semiring gives the expected return $J = 1.4$ of Chapter 1.
Max-sum gives 9, the return of the single best trajectory: start in cell 1,
move Right into cell 2, then move Left and slip, which keeps the robot at
the charger for free. The best trajectory counts on luck.

In [6]:
assert np.isclose(SumProduct.read(eliminate(SumProduct(), terms, order)), 1.0)
assert np.isclose(Expectation.read(eliminate(Expectation(), terms, order)), 1.4)
assert np.isclose(MaxSum.read(eliminate(MaxSum(), terms, order)), 9.0)

## Two steps of max-sum elimination, by hand (Section 2)

The last move of the track: eliminate the last state $s_2$, then the last
action $a_1$. A probability is lifted to 0 where it is positive and to
$-\infty$ where it is zero.

In [7]:
with np.errstate(divide="ignore"):
    possible = np.where(dynamics > 0, 0.0, -np.inf)  # lifted p(s2 | s1, a1)
print("lifted dynamics, rows (s1, a1), columns s2:\n", possible.reshape(6, 3))

# Step 1: multiply (add) with the final reward, then maximize over s2.
bucket_s2 = possible + final  # axes s1, a1, s2
phi_sa = bucket_s2.max(axis=2)
print("bucket of s2:\n", bucket_s2.reshape(6, 3))
print("new factor on (s1, a1), the best reachable final reward:\n", phi_sa)
print("the average instead (expectation semiring):\n", dynamics @ final)
assert np.allclose(phi_sa, [[0, 0], [0, 10], [10, 10]])

# Step 2: multiply (add) with the reward of the move, then maximize over a1.
bucket_a1 = reward + phi_sa  # axes s1, a1
phi_s = bucket_a1.max(axis=1)
print("bucket of a1:\n", bucket_a1)
print("new factor on s1:", phi_s, " best action:", bucket_a1.argmax(axis=1))
print("regret of each action:\n", bucket_a1 - phi_s[:, None])
assert np.allclose(bucket_a1, [[0, -1], [0, 9], [10, 9]])
assert np.allclose(phi_s, [0, 9, 10])
assert list(bucket_a1.argmax(axis=1)) == [L, R, L]

# The correct order for a decision: average over s2, then maximize over a1.
correct = (reward + dynamics @ final)
print("average over s2, then max over a1:", correct.max(axis=1),
      " best action:", correct.argmax(axis=1))
assert np.allclose(correct.max(axis=1), [0, 7, 9])
assert list(correct.argmax(axis=1)) == [L, R, R]

lifted dynamics, rows (s1, a1), columns s2:
 [[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0. -inf]
 [-inf   0.   0.]
 [-inf   0.   0.]
 [-inf -inf   0.]]
bucket of s2:
 [[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0. -inf]
 [-inf   0.  10.]
 [-inf   0.  10.]
 [-inf -inf  10.]]
new factor on (s1, a1), the best reachable final reward:
 [[ 0.  0.]
 [ 0. 10.]
 [10. 10.]]
the average instead (expectation semiring):
 [[ 0.  0.]
 [ 0.  8.]
 [ 2. 10.]]
bucket of a1:
 [[ 0. -1.]
 [ 0.  9.]
 [10.  9.]]
new factor on s1: [ 0.  9. 10.]  best action: [0 1 0]
regret of each action:
 [[ 0. -1.]
 [-9.  0.]
 [ 0. -1.]]
average over s2, then max over a1: [0. 7. 9.]  best action: [0 1 1]


## The tilted family (Section 3)

The tilted mean $\frac{1}{\kappa} \log \mathbb{E}[e^{\kappa R}]$ moves from
the worst return through the average to the best one as $\kappa$ grows.

In [8]:
returns = {}
for kappa in [-20, -2, -0.5, -0.01, 0.01, 0.5, 2, 20]:
    semiring = Tilted(kappa)
    returns[kappa] = float(semiring.read(eliminate(semiring, terms, order)))
    print(f"kappa = {kappa:6.2f}   tilted mean of the return = "
          f"{returns[kappa]:7.4f}")

kappa = -20.00   tilted mean of the return = -1.8502
kappa =  -2.00   tilted mean of the return = -0.9265
kappa =  -0.50   tilted mean of the return = -0.2768
kappa =  -0.01   tilted mean of the return =  1.3274
kappa =   0.01   tilted mean of the return =  1.4748
kappa =   0.50   tilted mean of the return =  5.4251
kappa =   2.00   tilted mean of the return =  7.6490
kappa =  20.00   tilted mean of the return =  8.8391


In [9]:
assert abs(returns[0.01] - 1.4) < 0.1 and abs(returns[-0.01] - 1.4) < 0.1
assert abs(returns[20] - 9.0) < 0.3  # approaches the best return
assert abs(returns[-20] + 2.0) < 0.3  # approaches the worst return

The soft maximum with temperature $\eta$ is the same rule with
$\kappa = 1 / \eta$: for $\eta = 2$ it gives the number of $\kappa = 0.5$.

In [10]:
eta = 2.0
soft = Tilted(1 / eta)
soft_maximum = float(soft.read(eliminate(soft, terms, order)))
print(f"soft maximum, eta = {eta:g}: {soft_maximum:.4f}")
assert np.isclose(soft_maximum, returns[0.5])

soft maximum, eta = 2: 5.4251


A smaller example of the tilted mean: a fair coin that pays 0 or 10.

In [11]:
coin_p, coin_v = np.array([0.5, 0.5]), np.array([0.0, 10.0])
for kappa in [-5, -0.5, -0.01, 0.01, 0.5, 5]:
    tilted = np.log(coin_p @ np.exp(kappa * coin_v)) / kappa
    print(f"kappa = {kappa:5.2f}   tilted mean = {tilted:.3f}")
assert np.isclose(np.log(coin_p @ np.exp(0.5 * coin_v)) / 0.5, 8.627, atol=1e-3)
assert np.isclose(np.log(coin_p @ np.exp(-0.5 * coin_v)) / -0.5, 1.373, atol=1e-3)

kappa = -5.00   tilted mean = 0.139
kappa = -0.50   tilted mean = 1.373
kappa = -0.01   tilted mean = 4.875
kappa =  0.01   tilted mean = 5.125
kappa =  0.50   tilted mean = 8.627
kappa =  5.00   tilted mean = 9.861


The best return is 9 and the worst is $-2$ (move right twice from cell 0 and
slip both times). A brute-force check over all trajectories:

In [12]:
trajectories = []
for s0 in range(3):
    for a0 in range(2):
        for s1 in range(3):
            for a1 in range(2):
                for s2 in range(3):
                    p = (prior[s0] * policy[s0, a0] * dynamics[s0, a0, s1] *
                         policy[s1, a1] * dynamics[s1, a1, s2])
                    if p > 0:
                        trajectories.append(
                            (p, reward[s0, a0] + reward[s1, a1] + final[s2]))
p, ret = np.array(trajectories).T
print("number of possible trajectories:", len(p))
print("sum of probabilities:", p.sum())
print("expected return:", p @ ret)
print("best and worst return:", ret.max(), ret.min())
for kappa in [-2, 0.5, 2]:
    brute = np.log(p @ np.exp(kappa * ret)) / kappa
    assert np.isclose(brute, returns[kappa])
    print(f"kappa = {kappa:4.1f}: brute force {brute:.4f}")

number of possible trajectories: 24
sum of probabilities: 1.0
expected return: 1.4
best and worst return: 9.0 -2.0
kappa = -2.0: brute force -0.9265
kappa =  0.5: brute force 5.4251
kappa =  2.0: brute force 7.6490


## Joint optimization is a member of the family (Section 2)

Trajectory optimization, as it is done with GTSAM, maximizes the return plus
the log-probability over all variables at once. That is max-sum with a
probability lifted to its logarithm. There is no policy: the actions are
free variables.

In [13]:
class MaxSumLog(MaxSum):
    """Max-sum with a probability f lifted to log f (minus infinity at 0)."""
    name = "max-sum, log lifting"

    def __init__(self, weight=1.0):
        self.weight = weight  # the weight of the reward factors

    def probability(self, p):
        with np.errstate(divide="ignore"):
            return (np.log(p),)

    def reward(self, r):
        return (self.weight * r,)


planning_terms = [term for term in terms if term[1] not in
                  [("s0", "a0"), ("s1", "a1")] or term[0] == "reward"]

best_possible = float(eliminate(MaxSum(), planning_terms, order)[0])
joint = float(eliminate(MaxSumLog(), planning_terms, order)[0])
print("max-sum, 0 / -inf lifting:", best_possible)
print("max-sum, log lifting     :", round(joint, 4))

# The trajectory that attains it, by brute force over the trajectories.
candidates = []
for s0 in range(3):
    for a0 in range(2):
        for s1 in range(3):
            for a1 in range(2):
                for s2 in range(3):
                    p = prior[s0] * dynamics[s0, a0, s1] * dynamics[s1, a1, s2]
                    if p > 0:
                        ret = reward[s0, a0] + reward[s1, a1] + final[s2]
                        candidates.append(
                            (np.log(p) + ret, ret, p, (s0, a0, s1, a1, s2)))
score, ret, p, trajectory = max(candidates)
print("best trajectory (s0, a0, s1, a1, s2):", trajectory,
      " return", ret, " probability", round(p, 3))
assert np.isclose(best_possible, 9.0)
assert np.isclose(joint, score) and np.isclose(joint, 8 + np.log(0.4))

max-sum, 0 / -inf lifting: 9.0
max-sum, log lifting     : 7.0837
best trajectory (s0, a0, s1, a1, s2): (1, 1, 2, 1, 2)  return 8.0  probability 0.4


On the line of Chapter 1 (linear dynamics with noise of variance 0.5,
quadratic rewards) the joint optimization is a least-squares problem. With a
weight $\kappa$ on the reward factors, the plan from a state $x$ minimizes
$\kappa\,(\text{costs}) + \sum w^2 / (2 \Sigma_w)$ over the actions and
the slips $w$. Its feedback gains are compared with the Riccati gains, and
each policy is then evaluated on the real, noisy system.

In [14]:
def joint_gains(weight, noise=0.5):
    """Gains of the joint least-squares plan, by its backward recursion.

    Maximizing over the slip w as well as over the action replaces the value
    matrix P by P / (1 + 2 * weight * P * noise) before each Riccati step.
    """
    P, gains = 1.0, []
    for _ in range(2):
        tilted = P / (1 + 2 * weight * P * noise)
        K = tilted / (1 + tilted)
        P = 1 + tilted - tilted ** 2 / (1 + tilted)
        gains.append(K)
    return gains[::-1]  # K0, K1


def least_squares_gain(weight, noise=0.5):
    """The first gain again, by solving the joint problem as one linear system.

    Unknowns z = (u0, x1, u1, x2), for x0 = 1; minimizes
    weight * (x0^2 + u0^2 + x1^2 + u1^2 + x2^2)
      + (x1 - x0 - u0)^2 / (2 noise) + (x2 - x1 - u1)^2 / (2 noise).
    """
    rows, rhs = [], []
    for index in range(4):  # the four cost terms on u0, x1, u1, x2
        row = np.zeros(4)
        row[index] = np.sqrt(weight)
        rows.append(row), rhs.append(0.0)
    scale = 1 / np.sqrt(2 * noise)
    rows.append(scale * np.array([-1.0, 1, 0, 0])), rhs.append(scale * 1.0)
    rows.append(scale * np.array([0.0, -1, -1, 1])), rhs.append(0.0)
    z = np.linalg.lstsq(np.array(rows), np.array(rhs), rcond=None)[0]
    return -z[0]  # u0 = -K0 x0 with x0 = 1


def true_return(K0, K1, noise=0.5):
    """Expected return of u = -K x on the real system, x0 ~ N(2, 1)."""
    P, beta = 1.0, 0.0
    for K in [K1, K0]:
        beta += P * noise
        P = 1 + K ** 2 + P * (1 - K) ** 2
    return -(P * (2 ** 2 + 1) + beta)


assert np.isclose(true_return(0.6, 0.5), -9.25)  # the Riccati gains
print("weight    K0      K1     true expected return")
print(f"   -> 0  0.6000  0.5000  {true_return(0.6, 0.5):.3f}   (Riccati)")
expected = {0.5: (0.4516, 0.4, -9.565), 1.0: (0.3636, 0.3333, -10.089),
            2.0: (0.2632, 0.25, -11.070)}
for weight, (K0_ref, K1_ref, J_ref) in expected.items():
    K0, K1 = joint_gains(weight)
    J = true_return(K0, K1)
    print(f"  {weight:4.1f}   {K0:.4f}  {K1:.4f}  {J:.3f}")
    assert np.isclose(K0, least_squares_gain(weight))
    assert np.allclose([K0, K1, J], [K0_ref, K1_ref, J_ref], atol=1e-3)
K0, K1 = joint_gains(1e-6)
assert np.allclose([K0, K1], [0.6, 0.5], atol=1e-4)

# With a tight dynamics factor the plan is the Riccati plan, whatever the weight.
K0, K1 = joint_gains(1.0, noise=1e-6)
print("weight 1, dynamics variance 1e-6:", round(K0, 4), round(K1, 4))
assert np.allclose([K0, K1], [0.6, 0.5], atol=1e-4)

weight    K0      K1     true expected return
   -> 0  0.6000  0.5000  -9.250   (Riccati)
   0.5   0.4516  0.4000  -9.565
   1.0   0.3636  0.3333  -10.089
   2.0   0.2632  0.2500  -11.070
weight 1, dynamics variance 1e-6: 0.6 0.5


## Conditionals and the normalization invariant (Section 5)

Take the bucket of the last action $a_1$ under the coin-flip policy. Its
product is $(\pi(a \mid s), Q_1(s, a))$. Dividing by the new factor gives
the conditional, and adding the conditional over the action gives the *one*
of the semiring, in every row.

In [15]:
Q1 = reward + dynamics @ final  # Q_1(s, a) = r(s, a) + sum_s' p(s'|s,a) r(s')
print("Q1 =\n", Q1)

# Expectation semiring: the surprise is the advantage, with mean zero.
V1 = (policy * Q1).sum(1, keepdims=True)
advantage = Q1 - V1
print("V1 =", V1.ravel())
print("advantage =\n", advantage)
print("sum of pi * advantage per cell:", (policy * advantage).sum(1))
assert np.allclose((policy * advantage).sum(1), 0)

# Max-sum: the surprise is the regret, with maximum zero.
regret = Q1 - Q1.max(1, keepdims=True)
print("regret =\n", regret)
assert np.allclose(regret.max(1), 0)

# Tilted, kappa = 1: the surprise is the soft advantage, with tilted mean zero.
kappa = 1.0
soft_V1 = np.log((policy * np.exp(kappa * Q1)).sum(1, keepdims=True)) / kappa
soft_advantage = Q1 - soft_V1
tilted_policy = policy * np.exp(kappa * soft_advantage)
print("soft V1 =", soft_V1.ravel())
print("tilted policy pi * exp(kappa * soft advantage) =\n", tilted_policy)
print("its rows sum to", tilted_policy.sum(1))
assert np.allclose(tilted_policy.sum(1), 1)

# With a uniform policy, the tilted policy is the softmax of kappa * Q, and
# the soft value is the log-sum-exp of kappa * Q (up to the constant log 2).
assert np.allclose(tilted_policy, softmax(kappa * Q1, axis=1))
assert np.allclose(soft_V1.ravel(),
                   (logsumexp(kappa * Q1, axis=1) - np.log(2)) / kappa)

Q1 =
 [[ 0. -1.]
 [ 0.  7.]
 [ 2.  9.]]
V1 = [-0.5  3.5  5.5]
advantage =
 [[ 0.5 -0.5]
 [-3.5  3.5]
 [-3.5  3.5]]
sum of pi * advantage per cell: [0. 0. 0.]
regret =
 [[ 0. -1.]
 [-7.  0.]
 [-7.  0.]]
soft V1 = [-0.3799  6.3078  8.3078]
tilted policy pi * exp(kappa * soft advantage) =
 [[0.7311 0.2689]
 [0.0009 0.9991]
 [0.0009 0.9991]]
its rows sum to [1. 1. 1.]


## The log-dual form (Section 6)

In the form $(\ell, v)$, with $\ell = \log p$, multiplication adds both
channels. A product of many small probabilities then stays finite.

In [16]:
def log_dual_plus(l, v, axis):
    """Add (l, v) entries over an axis: log-sum-exp and a weighted mean."""
    total = logsumexp(l, axis=axis)
    weights = np.exp(l - np.expand_dims(total, axis))
    return total, (weights * v).sum(axis)


# 400 unnormalized factors on one binary variable, each with a small
# probability and a reward of 1 for the second outcome.
p = np.array([1e-3, 2e-3])
r = np.array([0.0, 1.0])
n = 400

# Stored form (p, w): the product underflows to zero.
p_product = p ** n
w_product = p_product * (n * r)
with np.errstate(invalid="ignore", divide="ignore"):
    print("(p, w):", w_product.sum() / p_product.sum())

# Log-dual form: both channels add.
l_product, v_product = n * np.log(p), n * r
total, value = log_dual_plus(l_product, v_product, axis=0)
print("(l, v): log p =", total, " value =", value)
assert np.isclose(value, 400.0)

(p, w): nan
(l, v): log p = -2485.8432393688768  value = 400.0
